<a href="https://colab.research.google.com/github/RMoulla/PYK_Lyon/blob/main/TP_Seloger_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Ouvrir dans Colab"/></a>


# Travaux pratiques : analyse des prix des biens immobiliers.

Étude d'un jeu de données comportant des informations immobilières provenant du site seloger.com.

Dans un premier temps, nous nous proposons de faire une étude exploratoire pour se familiariser avec la donnée et comprendre comment les variables se comportent. Cette étape permettra egalement de débusquer d'eventuelles anomalies et de nettoyer la donnée si nécessaire.

Dans un second temps, nous nous attacherons à l'explicabilité des données, c'est à dire répondre à la question "Quelles sont les variables les plus importantes pour expliquer le prix d'un bien ?"

Finalement, nous entrainerons des modèles de Machine Learning sur des données d'entrainement et nous prédirons sur des données de test afin d'évaluer les performances du modèles.

## Description des variables

- `number` : rang de l'annonce dans la collecte, exactement identique à `position`.
- `codeinsee` : code INSEE de la commune où se situe le bien.
- `codepostal` : code postal du bien, exactement identique à `cp`.
- `cp` : code postal du bien, exactement identique à `codepostal`.
- `etage` : étage du bien ; la variable vaut toujours 0 dans ce jeu de données.
- `idagence` : identifiant de l'agence immobilière.
- `idannonce` : identifiant de l'annonce immobilière.
- `idtiers` : identifiant du tiers qui publie l'annonce.
- `idtypechauffage` : code du type de chauffage ; la variable vaut toujours 0 dans ce jeu de données.
- `idtypecommerce` : code du type de transaction ; la variable vaut toujours 0 dans ce jeu de données.
- `idtypecuisine` : type de cuisine du bien.
- `idtypepublicationsourcecouplage` : code de la source de publication ou du couplage de l'annonce.
- `naturebien` : code catégoriel décrivant la nature du bien.
- `nb_chambres` : nombre de chambres.
- `nb_photos` : nombre de photos associées à l'annonce.
- `nb_pieces` : nombre de pièces du bien.
- `position` : rang de l'annonce dans la collecte, exactement identique à `number`.
- `prix` : prix affiché du bien.
- `si_balcon` : indicateur binaire de la présence d'un balcon.
- `si_sdEau` : indicateur de la présence d'une salle d'eau ; la variable vaut toujours 0 dans ce jeu de données.
- `si_sdbain` : indicateur de la présence d'une salle de bains ; la variable vaut toujours 0 dans ce jeu de données.
- `surface` : surface du bien en mètres carrés.
- `typedebien` : type de bien immobilier.
- `ville` : ville ou arrondissement où se situe le bien.

In [ ]:
!python --version

In [ ]:
# Librairie pour lire des fichiers de données tabulaires
import pandas as pd

In [ ]:
pd.set_option("display.max_columns", 25)
dataset = pd.read_csv('selogerdata.csv')

In [ ]:
dataset.shape

Le jeu de données contient 8899 lignes et 24 colonnes.

In [ ]:
# Visualiser les 5 premières lignes du dataset
dataset.head()

In [ ]:
dataset.describe()

## Nettoyage des données

On peut d'ores et déjà remarquer que les colonnes `idtypecyuisine` et `ville` doivent etre nettoyées

In [ ]:
# Afficher le nom des colonnes du dataset
dataset.columns

In [ ]:
# Afficher la fréquence des types de cuisine.
dataset['idtypecuisine'].unique()

Que veulent dire les valeurs 0 et -1 ?
On pourrait penser que 0 corresond à la situation ou il n'y a pas de cuisine mais ce n'est pas le cas en regardant les données. Nous allons créer des groupes à part pour ces catégories


In [ ]:
# Correction des données
mapper = {
    's√©par√©e': 'séparée',
    '0': 'categorie_0',
    's√©par√©e √©quip√©e': 'séparée équipée',
    'am√©ricaine √©quip√©e': 'américaine équipée',
    '√©quip√©e': 'équipée',
    'aucune' : 'aucune',
    'am√©ricaine': 'américaine',
    'coin cuisine': 'coin cuisine',
    'coin cuisine √©quip√©': 'coin cuisine équipé',
    '-1': 'catégorie_1'
}

In [ ]:
# Appliquer les corrections
dataset['idtypecuisine'] = dataset['idtypecuisine'].map(mapper)

Nous allons également corriger les caractères accentués pour la variable ville.

In [ ]:
dataset['ville'].unique()

In [ ]:
# Corriger les accents
dataset['ville'] =  (
    dataset['ville']
    .str.replace('√®', 'è')
    .str.replace('√â', 'É')
    .str.replace('√©', 'é')
    .str.replace('√¨', 'ê')
)

In [ ]:
dataset['ville'].unique()

## Analyse exploratoire des données
Nous allons commencer par un aperçu sur les données sous forme de résumé statitiques des différentes variables numériques.

In [ ]:
dataset.describe()

Ce résumé statistique comporte déjà plusieurs enseignements :

* Les colonnes `etage`, `idtypechauffage`, `idtypecommerce`, `si_sdEau`,	`si_sdbain` ne comportent que des valeurs nulles. Elles n'apportent donc aucune information pour expliquer ou prédire le prix.

* Le jeu de données contient des outliers : on peut par exemple noter que le maximum du nombre de pièces est 53. De même le maximum du nombre de chambre est 22. Par ailleurs le minimum de la surface est 0.

Regardons ces outliers plus en détails.


In [ ]:
# sélectionner la ligne qui correspond au bien ayant 53 pièces
dataset[dataset['nb_pieces'] == 53]

In [ ]:
# Sélectionner la ligne qui correspond au bien ayant 22 chambres

dataset[dataset['nb_chambres'] == 22]

In [ ]:
# Sélectionner la ligne qui correspond au bien dont la surface est nulle

dataset[dataset['surface'] == 0]

Il s'agit clairement de valeurs aberrantes, que l'on va enlever.

In [ ]:
# Supprimer les lignes aberrantes

dataset.drop([1236, 1318, 2030], axis = 0, inplace = True)

Ragrdons maintenant la cardinalité des différentes variables. Ce qui nous intéresse ici, ce sont les variables catégorielles.

In [ ]:
# Combien de valeurs uniques possède chaque colonne ?
dataset.nunique().sort_values()

* Les variables `si_sdbain`, `si_sdEau`, `etage`, `idtypechauffage`, `idtypecommerce` ne comportent qu'une unique valeur, comme nous venons de le voir plus haut.
* `cp` et `codepostal` semblent se référer à la même variable. Par ailleurs, `codeinsee` et `ville` ont l'air de vehiculer plus ou moins la même information. En première approximation nous ne garderons qu'une seule de ces 4 variables.

* Les variables `position` et `number` ont le même nombre de modalités.

* Plus important peut-être, il y a 3815 modalités pour `idannonce` (annonces différentes) alors que le jeu de données contient 8899 lignes. On peut soupçonner ici la présence de doublons.

* Les variables `idagence` et`idtiers` ont le même nombre de modalités, très élevé. Il y a 1413 modalités pour chacune des variables alors qu'il n'y a potentiellement que 3815 annonces uniques, soit un peu plus de deux annonces par modalité de `idagence`.

Regardons ce dernier point plus en détails.



In [ ]:
idanonce_counts = dataset['idannonce'].value_counts()
idanonce_counts.values[:100]

On peut noter la présence d'annonces qui se répètent 202 ou 53 fois. Il s'agit probablement de doublons dus à la fusions de plusieurs bases de données.

In [ ]:
# Supprimer les doublons
dataset.drop_duplicates(inplace = True)
dataset.shape

In [ ]:
# Les colonnes sont identiques, on ne va en garder qu'une seule.
dataset['cp'].equals(dataset['codepostal'])

In [ ]:
# Les colonnes sont identiques, on ne va en garder qu'une seule.
dataset['position'].equals(dataset['number'])

Nous allons maintnant faire le bilan des variables à supprimer.

* Les variables `si_sdbain`, `si_sdEau`, `etage`, `idtypechauffage`, `idtypecommerce` ne comportent que des valeurs nulles.
* Les variables `idtiers` et `idagence` renvoient vers la même information. Par ailleurs, elles comportent beaucoup de trop de modalités par rapport au nombre d'oberservations.
* Les variables `cp`, `codepostal` sont identiques et contiennent à peu près la même information que `codeinsee`. Cette information est déjà présente dans la viable `ville`.

In [ ]:
# Bilan des variables à supprimer
to_remove = [
    'si_sdbain', 'si_sdEau', 'etage',
    'idtypechauffage', 'idtypecommerce', 'codepostal',
    'number', 'idtiers', 'cp', 'codeinsee', 'idannonce', 'idagence']



In [ ]:
# Enlever les colonnes dans to_remove

dataset.drop(to_remove, axis = 1, inplace = True)
dataset.shape

Il est également nécessaire de distinguer entre les variables quantitatives et les variables catégorielles, qui seront analysées de manières différentes.

In [ ]:
categorical = [
    'si_balcon', 'typedebien', 'idtypepublicationsourcecouplage',
    'naturebien','idtypecuisine', 'ville'
]

numerical = [
    'nb_chambres', 'nb_pieces', 'nb_photos', 'surface', 'position'
]


### Variables quantitatives

Dans ce qui suit, nous allons analyser les distributions des variables ainsi que les corrélations entre les différentes variables explicatives abvec la variable `prix`. Nous allons utiliser ici les deux packages de visualisation en Python `matplotlib` et `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

Commençons d'abord par regarder la distribution de la variable d'intérêt `prix`.


In [ ]:
# densité de la variable prix sous forme de disribution

sns.kdeplot(data = dataset, x = 'prix', fill=True)

In [ ]:
# Visualiser la surface en histogramme

sns.histplot(dataset['surface'])

In [ ]:
# Visualiser la variable si_balcon en countplot

sns.countplot(data=dataset, x='si_balcon')


In [ ]:
# Visualiser le prix en fonction de la surface

sns.scatterplot(data=dataset, x='surface', y='prix')

In [ ]:
# densité de la variable prix en fonction de si_balcon sous forme de disribution

sns.kdeplot(data = dataset, x = 'prix', hue='si_balcon', fill=True)

In [ ]:
# Visualiser le prix en fonction de si_balcon en boxplot

sns.boxplot(data=dataset, x='si_balcon', y='prix')

Regardons maintenant les distributions des différentes variables numériques.

In [ ]:
plt.rcParams["figure.figsize"]

fig, axs = plt.subplots(1,5, figsize=(18,3))
for i, feature in enumerate(numerical):
    sns.histplot(dataset[feature], ax = axs[i])
plt.show()

Nous allons étudier maintenant les corrélation entre les variables numériques et la variable d'intérêt `prix`.

In [ ]:
fig, axs = plt.subplots(1,5, figsize=(18,3))
for i, feature in enumerate(numerical):
    sns.scatterplot(x = dataset[feature], y = dataset['prix'], ax = axs[i])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(y=dataset['prix'], x=dataset['position'], hue=dataset['position'], ax=ax, palette='Set1', legend=False)
ax.tick_params(axis='x', rotation=90)
plt.show()

In [ ]:
# Tracer un corrélogramme des variables numériques
sns.heatmap(dataset[numerical].corr())

### Variables catégorielles

On peut commencer pas visualiser les variables qui ont un faible nombre de modalités en fonction de la variable `prix`.

In [ ]:
categorical_sub = ['si_balcon', 'typedebien', 'idtypepublicationsourcecouplage', 'naturebien']
fig, axs = plt.subplots(1, 4, figsize=(18,3))
for i, feature in enumerate(categorical_sub):
    sns.kdeplot(data = dataset, x="prix", hue = feature, fill=True, ax = axs[i])

plt.show()

Regardons maintenant les variables catégorielles ayant un nombre imortant de modalités. Commençons par la variable `ville`.

In [ ]:
# Afficher le nombre de modalités de la variable ville
dataset['ville'].nunique()

In [ ]:
# Répartition des observations sur les modalités
dataset['ville'].value_counts()

In [ ]:
dataset.loc[~dataset['ville'].str.startswith('Paris '), 'ville'] = 'Banlieue'
dataset['ville'].value_counts()

Nous voyons ici que les villes situées en dehors de Paris sont très faiblement représentées dans le jeu de données. Nous allons les regrouper dans une seule modalité "banlieue".

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(y=dataset['prix'], x=dataset['ville'], hue=dataset['ville'], ax=ax, palette='Set1', legend=False)
ax.tick_params(axis='x', rotation=90)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(y=dataset['prix'], x=dataset['idtypecuisine'], hue=dataset['idtypecuisine'], ax=ax, palette='Set1', legend=False)
ax.tick_params(axis='x', rotation=90)
plt.show()

In [ ]:
dataset.head()

# Approche explicative : étude des déterminants du prix

On va maintenant essayer de comprendre quelles sont les variables les plus déterminantes qui exliquent le prix d'un bien immobilier. Pour ce faire, nous allons utiliser un modèle de régression linéaire. Mais il nous reste encore une étape : transformer les variables catégorielles en variables numériques.

In [ ]:
dataset.columns

In [ ]:
# Transformer les variables catégorielles en variables numériques avec du one-hot encoding il ne reste que des variables numérique
categorical_sub = [
     'typedebien', 'idtypepublicationsourcecouplage',
    'naturebien','idtypecuisine', 'ville'
]


df = pd.get_dummies(dataset, columns = categorical_sub, drop_first=True).astype(int)
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# normalisation des données

scaler = StandardScaler()
df[numerical] = scaler.fit_transform(df[numerical])
df.head()

In [ ]:
df.shape

## Régression linéaire

In [ ]:
import statsmodels.api as sm

# Définir la variable dépendante (label) y et les variables explicatives X
y = df['prix']
X = df.drop(['prix'], axis=1)

# Ajouter une constante à la matrice de design X
X = sm.add_constant(X)

# Ajuster un modèle linéaire
model = sm.OLS(y, X).fit()

# Afficher les informations sur le modèle
print(model.summary())


In [ ]:
# Sélectionner les variables ayant une p-valeue inférieures à 0.05

selected_columns = X.columns[model.pvalues < 0.05]
selected_columns



In [ ]:
# Ajuster une régression linéaire avec les variables sélectionnées

X = df[selected_columns]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

# Prédire les prix

On se propose maintenant de prédire les prix des biens immobliers.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
import numpy as np
# Séparer les données en données d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entraîner le modèle de régression linéaire sur les données d'entraînement
model = sm.OLS(y_train, X_train).fit()

# Prédire les prix des biens immobiliers sur les données de test
y_pred = model.predict(X_test)

# Évaluer les performances du modèle
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean squared error: {rmse:.2f}")


In [ ]:


ecarts = np.abs(y_pred - y_test)

fig, ax = plt.subplots(figsize=(15, 5))
sns.histplot(ecarts, ax=ax, color='green', kde=False)
ax.set_title('Distribution des écarts')
plt.show()
